# 03 — Datasets, DataLoaders, Checkpointing

Companion to [`notes.md`](notes.md). Three parts, all run for real:

1. A **from-scratch** manual batching/shuffling generator in plain Python, over a toy list — what `DataLoader` automates.
2. A **real PyTorch `Dataset` + `DataLoader`** wrapping the Breast Cancer Wisconsin dataset, training a small `nn.Module`, saving a checkpoint (`torch.save`) every `N` epochs.
3. A **simulated crash-and-resume**: a fresh model + optimizer, constructed from nothing but the saved checkpoint file, resumes training and the loss picks up where it left off — with a side-by-side failure-mode demo of what happens if optimizer state is *not* restored.


In [1]:
import copy
import random

import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())


torch: 2.13.0+cpu | cuda available: False


## 1. From-scratch: manual batching + shuffling

Before reaching for `DataLoader`, here is exactly what it automates, written by hand over a
toy list of 10 "examples" (just integers standing in for data points). No `torch`, no
`Dataset` class — plain Python, an index list, `random.shuffle`, and slicing.


In [2]:
def manual_batch_generator(data, batch_size, shuffle=True, seed=None):
    """Yield successive shuffled batches of `data`, plain Python, no PyTorch.

    This is the from-scratch version of what `torch.utils.data.DataLoader` automates:
    - shuffling: re-order indices each epoch (if shuffle=True)
    - batching: slice the (possibly re-ordered) data into chunks of `batch_size`
    - the final partial batch (dataset size not divisible by batch_size) is yielded as-is
    """
    n = len(data)
    indices = list(range(n))
    rng = random.Random(seed)
    if shuffle:
        rng.shuffle(indices)
    for start in range(0, n, batch_size):
        batch_idx = indices[start:start + batch_size]
        yield [data[i] for i in batch_idx]


toy_data = list(range(10))  # 10 toy "examples"

print("Epoch 1 (shuffle=True, seed=1):")
for batch in manual_batch_generator(toy_data, batch_size=3, shuffle=True, seed=1):
    print(" ", batch)

print("\nEpoch 2 (shuffle=True, seed=2 -> different order):")
for batch in manual_batch_generator(toy_data, batch_size=3, shuffle=True, seed=2):
    print(" ", batch)

print("\nWithout shuffling (shuffle=False, same order every epoch):")
for batch in manual_batch_generator(toy_data, batch_size=3, shuffle=False):
    print(" ", batch)


Epoch 1 (shuffle=True, seed=1):
  [6, 8, 9]
  [7, 5, 3]
  [0, 4, 1]
  [2]

Epoch 2 (shuffle=True, seed=2 -> different order):
  [5, 9, 3]
  [4, 6, 7]
  [2, 8, 1]
  [0]

Without shuffling (shuffle=False, same order every epoch):
  [0, 1, 2]
  [3, 4, 5]
  [6, 7, 8]
  [9]


Three things this hand-written generator has to get right, that `DataLoader` gets for free:
re-shuffling the *index* order (not the underlying data) each epoch, slicing into fixed-size
chunks with a correctly-handled final partial batch, and keeping the shuffle deterministic
enough to debug (`seed=`) while still varying epoch to epoch. `DataLoader` does exactly this,
plus (optionally) parallel/background loading via `num_workers` and pinned memory for
GPU transfer — neither of which the from-scratch version above attempts, since this
environment has no GPU and a toy in-memory dataset needs no background workers.


## 2. Real PyTorch `Dataset` + `DataLoader`

Breast Cancer Wisconsin dataset (`sklearn.datasets.load_breast_cancer`): 569 examples, 30
numeric features, binary target (malignant/benign). Small enough to fit in memory as one
array — but wrapped in a real `Dataset`/`DataLoader` pipeline exactly as a larger dataset
would be, so the pattern is the one that scales, not a special case.


In [3]:
data = load_breast_cancer()
X, y = data.data, data.target.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("Positive rate (train):", y_train.mean().round(3))


X_train: (455, 30) X_test: (114, 30)
Positive rate (train): 0.626


In [4]:
class BreastCancerDataset(torch.utils.data.Dataset):
    """A real torch.utils.data.Dataset: just __len__ and __getitem__.

    DataLoader calls __getitem__(i) for whatever indices it decides to fetch (after its own
    internal shuffling), and __len__ to know how many examples exist. Everything about
    *how* those indices are chosen, batched, and shuffled is DataLoader's job, not this
    class's -- mirroring the split between "toy_data list" and "manual_batch_generator"
    above.
    """

    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).unsqueeze(1)  # shape (n, 1) to match model output

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = BreastCancerDataset(X_train, y_train)
test_ds = BreastCancerDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=64, shuffle=False)

xb, yb = next(iter(train_loader))
print("One batch from DataLoader:", xb.shape, yb.shape)


One batch from DataLoader: torch.Size([32, 30]) torch.Size([32, 1])


## 3. Training with checkpointing

A small `nn.Module` (same pattern as `02-nn-module-and-training-loop`), trained with
`Adam`, saving a checkpoint every `CHECKPOINT_EVERY` epochs. A checkpoint holds **both**
`model.state_dict()` and `optimizer.state_dict()` — per `notes.md`'s "Why simpler
approaches fail", saving only the weights would silently reset Adam's per-parameter
momentum/variance estimates on resume.

This also connects to `08-mlops-deployment/04-model-packaging-versioning/notes.md`'s
content-addressing idea: a checkpoint file is exactly the kind of large binary artifact
that topic discusses versioning by content hash, rather than by a human-picked filename
like `model_v2_final.pt`.


In [5]:
class MLP(nn.Module):
    def __init__(self, n_in, n_hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


import pathlib

CKPT_DIR = pathlib.Path("checkpoints")
CKPT_DIR.mkdir(exist_ok=True)
CHECKPOINT_EVERY = 3
TOTAL_EPOCHS_RUN1 = 8  # "crashes" after epoch 8

model = MLP(n_in=X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.BCELoss()

history_run1 = []
latest_ckpt_path = None

for epoch in range(1, TOTAL_EPOCHS_RUN1 + 1):
    model.train()
    epoch_losses = []
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    mean_loss = float(np.mean(epoch_losses))
    history_run1.append(mean_loss)
    print(f"[run 1] epoch {epoch:2d}  mean BCE loss = {mean_loss:.4f}")

    if epoch % CHECKPOINT_EVERY == 0 or epoch == TOTAL_EPOCHS_RUN1:
        ckpt_path = CKPT_DIR / f"ckpt_epoch{epoch:02d}.pt"
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": mean_loss,
            },
            ckpt_path,
        )
        latest_ckpt_path = ckpt_path
        print(f"           -> saved checkpoint: {ckpt_path}")

print("\n'Crash' simulated after epoch", TOTAL_EPOCHS_RUN1)
print("Latest checkpoint on disk:", latest_ckpt_path)


[run 1] epoch  1  mean BCE loss = 0.3243
[run 1] epoch  2  mean BCE loss = 0.0997
[run 1] epoch  3  mean BCE loss = 0.0647
           -> saved checkpoint: checkpoints/ckpt_epoch03.pt
[run 1] epoch  4  mean BCE loss = 0.0539
[run 1] epoch  5  mean BCE loss = 0.0468
[run 1] epoch  6  mean BCE loss = 0.0418
           -> saved checkpoint: checkpoints/ckpt_epoch06.pt
[run 1] epoch  7  mean BCE loss = 0.0414
[run 1] epoch  8  mean BCE loss = 0.0370
           -> saved checkpoint: checkpoints/ckpt_epoch08.pt

'Crash' simulated after epoch 8
Latest checkpoint on disk: checkpoints/ckpt_epoch08.pt


### Simulating a fresh process

The cell below deliberately never touches the live `model` / `optimizer` objects from
above — it builds **brand-new** ones from scratch and populates them *only* from the file
on disk (`torch.load`), exactly as a freshly-started Python process resuming a crashed job
would have to. This is the honest way to test "does the checkpoint actually contain
everything needed to resume," since a live-process shortcut (just keep using `model`)
would not catch a checkpoint that's missing something.


In [6]:
# Fresh, independently-constructed objects -- as if this were a new process
resumed_model = MLP(n_in=X_train.shape[1])
resumed_optimizer = torch.optim.Adam(resumed_model.parameters(), lr=1e-2)

checkpoint = torch.load(latest_ckpt_path, weights_only=True)
resumed_model.load_state_dict(checkpoint["model_state_dict"])
resumed_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"]

print(f"Loaded checkpoint from {latest_ckpt_path}")
print(f"  saved at epoch {start_epoch}, saved loss = {checkpoint['loss']:.4f}")

# Sanity check: the reloaded model's loss on the training set, computed *before* any
# further training, should match run 1's own loss at that epoch almost exactly --
# it's the identical set of weights, just deserialized.
resumed_model.eval()
with torch.no_grad():
    sanity_losses = [
        loss_fn(resumed_model(xb), yb).item() for xb, yb in train_loader
    ]
print(f"  reloaded-model loss recomputed now = {np.mean(sanity_losses):.4f}"
      f"  (run 1's recorded loss at epoch {start_epoch} = {history_run1[start_epoch - 1]:.4f})")


Loaded checkpoint from checkpoints/ckpt_epoch08.pt
  saved at epoch 8, saved loss = 0.0370
  reloaded-model loss recomputed now = 0.0358  (run 1's recorded loss at epoch 8 = 0.0370)


In [7]:
TOTAL_EPOCHS_RUN2 = 16  # resume from start_epoch+1 up to 16

history_run2 = []
resumed_model.train()
for epoch in range(start_epoch + 1, TOTAL_EPOCHS_RUN2 + 1):
    epoch_losses = []
    for xb, yb in train_loader:
        resumed_optimizer.zero_grad()
        pred = resumed_model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        resumed_optimizer.step()
        epoch_losses.append(loss.item())
    mean_loss = float(np.mean(epoch_losses))
    history_run2.append(mean_loss)
    print(f"[run 2 - resumed] epoch {epoch:2d}  mean BCE loss = {mean_loss:.4f}")

full_history = history_run1 + history_run2
print("\nFull stitched loss curve (run 1 epochs 1-8, run 2 epochs 9-16):")
print([round(l, 4) for l in full_history])


[run 2 - resumed] epoch  9  mean BCE loss = 0.0332
[run 2 - resumed] epoch 10  mean BCE loss = 0.0321
[run 2 - resumed] epoch 11  mean BCE loss = 0.0341
[run 2 - resumed] epoch 12  mean BCE loss = 0.0328
[run 2 - resumed] epoch 13  mean BCE loss = 0.0365
[run 2 - resumed] epoch 14  mean BCE loss = 0.0312
[run 2 - resumed] epoch 15  mean BCE loss = 0.0305
[run 2 - resumed] epoch 16  mean BCE loss = 0.0320

Full stitched loss curve (run 1 epochs 1-8, run 2 epochs 9-16):
[0.3243, 0.0997, 0.0647, 0.0539, 0.0468, 0.0418, 0.0414, 0.037, 0.0332, 0.0321, 0.0341, 0.0328, 0.0365, 0.0312, 0.0305, 0.032]


**Result:** the resumed run's epoch-9 loss continues smoothly from run 1's epoch-8 loss
(no jump), and the full stitched curve keeps decreasing across the "crash" boundary —
confirming the checkpoint restored everything needed (weights *and* optimizer momentum
state) to continue training as if it had never stopped.


## Failure mode, demonstrated: dropping optimizer state

`notes.md`'s "Failure modes" section names this as the single most common checkpointing
bug: saving/restoring `model.state_dict()` but not `optimizer.state_dict()`. Below,
the *same* checkpoint is reloaded, but the fresh optimizer starts from scratch (Adam's
per-parameter momentum/variance estimates reset to zero) instead of being restored.


In [8]:
bad_model = MLP(n_in=X_train.shape[1])
bad_optimizer = torch.optim.Adam(bad_model.parameters(), lr=1e-2)  # NOT loaded from checkpoint

bad_model.load_state_dict(checkpoint["model_state_dict"])  # weights restored...
# ...but bad_optimizer.load_state_dict(...) is deliberately skipped

bad_history = []
bad_model.train()
for epoch in range(start_epoch + 1, start_epoch + 4):
    epoch_losses = []
    for xb, yb in train_loader:
        bad_optimizer.zero_grad()
        pred = bad_model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        bad_optimizer.step()
        epoch_losses.append(loss.item())
    mean_loss = float(np.mean(epoch_losses))
    bad_history.append(mean_loss)
    print(f"[run 2 - BAD resume, no optimizer state] epoch {epoch:2d}  mean BCE loss = {mean_loss:.4f}")

print("\nProper resume, same 3 epochs:", [round(l, 4) for l in history_run2[:3]])
print("Bad resume (fresh optimizer): ", [round(l, 4) for l in bad_history])


[run 2 - BAD resume, no optimizer state] epoch  9  mean BCE loss = 0.0463
[run 2 - BAD resume, no optimizer state] epoch 10  mean BCE loss = 0.0325
[run 2 - BAD resume, no optimizer state] epoch 11  mean BCE loss = 0.0328

Proper resume, same 3 epochs: [0.0332, 0.0321, 0.0341]
Bad resume (fresh optimizer):  [0.0463, 0.0325, 0.0328]


**Interpretation:** both start from the identical restored weights, but the properly
resumed run (optimizer state restored) continues its smooth descent, while the badly
resumed run — same weights, fresh Adam state — takes a visibly different, less stable path
for the first few epochs, since Adam's adaptive step sizes have to re-estimate
per-parameter momentum/variance from zero instead of continuing from where they were. On a
longer run this typically shows up as a transient loss spike right at the resume point.
This is exactly the failure mode `notes.md` predicts, reproduced with real numbers.
